# 数据集预览 + LIF / PLIF / MSF 重建示意（$T=4$）

**说明（方法口径）**

1. **输入**：从 PlantVillage 取 batch，反归一化到 $[0,1]$ 得到强度图，再用 **TTFS（`first_spike_coding`）** 得到 `[T,B,3,H,W]`。
2. **“重建”**：将上述脉冲序列作为 **输入电流**（逐时间步送入），分别经过 **一层** `LIFNode` / `ParametricLIFNode` / `MSFNode`（与 `TrainConfig` 默认动力学一致），将 **输出脉冲在时间上取平均** 得到 `[B,3,H,W]`，作为与原始强度可比的 **空间重建图**（用于分布对比与可视化；**非**训练好的图像自编码器）。
3. **PlantVillage 为静态图**；“视频”小节用 **同一 batch 内多张图沿时间维当作帧** 的示意条带，仅作展示。

**运行（默认 Kaggle）**：先装依赖 → **clone / 解压仓库** → `chdir` 到仓库根 → 修改 **`DATA_ROOT`**（与 `kaggle_plantvillage_snn.ipynb` 中数据集路径一致）。本地 HF 可将下方 `USE_KAGGLE` 改为 `False`。

In [ ]:
# --- 依赖 ---
!pip -q install matplotlib spikingjelly torch torchvision
# 仅当 USE_KAGGLE=False 走 HuggingFace 时需要：
# !pip -q install datasets

## Kaggle：仓库与工作目录

先 **Add Input** 本仓库，或 **git clone** 到 `/kaggle/working/plantvillage_snn`。下一格若检测到该目录则自动 `chdir`。

In [ ]:
import os
from pathlib import Path

# os.chdir("/kaggle/working")
# !git clone --depth 1 --branch master https://github.com/xing11234/plantvillage_snn.git plantvillage_snn

KG_REPO = Path("/kaggle/working/plantvillage_snn")
if KG_REPO.is_dir():
    os.chdir(KG_REPO)
    print("cwd:", os.getcwd())
else:
    print("WARN: 未找到", KG_REPO, "— 请先 clone 或解压仓库到该路径。")

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from spikingjelly.activation_based import functional as sj_functional
from spikingjelly.activation_based import neuron

# 仓库根：在 notebooks/ 下运行时向上一级
REPO = Path.cwd().resolve()
if (REPO / "utils" / "train_utils.py").is_file():
    pass
elif (REPO.parent / "utils" / "train_utils.py").is_file():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from models.msf_neuron import MSFNode
from utils.train_utils import denormalize_to_01, first_spike_coding

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T = 4
torch.manual_seed(42)

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def denorm_batch(x: torch.Tensor) -> torch.Tensor:
    x = x.detach().cpu().float()
    return (x * _STD + _MEAN).clamp(0.0, 1.0)


def preview_dataset_grid(
    images: torch.Tensor,
    labels=None,
    *,
    num_cols: int = 4,
    num_rows: int = 2,
    figsize_per_cell: float = 2.5,
    title_prefix: str = "",
):
    n_show = min(num_cols * num_rows, images.size(0))
    ims = denorm_batch(images[:n_show])
    labs = labels[:n_show].tolist() if labels is not None else None
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(figsize_per_cell * num_cols, figsize_per_cell * num_rows))
    axes = np.atleast_2d(axes)
    for i in range(num_rows * num_cols):
        r, c = divmod(i, num_cols)
        ax = axes[r, c]
        if i < n_show:
            ax.imshow(ims[i].permute(1, 2, 0).numpy())
            t = f"{title_prefix}#{i}"
            if labs is not None:
                t += f"  y={labs[i]}"
            ax.set_title(t, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def build_neuron_layers(device: torch.device):
    lif = neuron.LIFNode(
        tau=2.0,
        decay_input=True,
        v_reset=0.0,
        detach_reset=True,
        step_mode="m",
    ).to(device)
    plif = neuron.ParametricLIFNode(
        init_tau=2.0,
        decay_input=True,
        v_reset=0.0,
        detach_reset=True,
        step_mode="m",
    ).to(device)
    msf = MSFNode(
        decay=0.25,
        D=4,
        v_threshold=1.0,
        surrogate="rect",
        surrogate_alpha=1.0,
        step_mode="m",
    ).to(device)
    return {"LIF": lif, "PLIF": plif, "MSF": msf}


def forward_neuron(layer: torch.nn.Module, x_seq: torch.Tensor) -> torch.Tensor:
    """x_seq [T,B,C,H,W] -> out [T,B,C,H,W]"""
    if isinstance(layer, (neuron.LIFNode, neuron.ParametricLIFNode)):
        sj_functional.reset_net(layer)
    return layer(x_seq)


def spike_rate_recon(out: torch.Tensor) -> torch.Tensor:
    """[T,B,C,H,W] -> [B,C,H,W] 时间平均发放率"""
    return out.float().mean(dim=0).clamp(0.0, 1.0)


def mse_vs_original(orig01: torch.Tensor, recon: torch.Tensor) -> float:
    """orig01, recon: [B,3,H,W] in [0,1]"""
    return float(F.mse_loss(recon, orig01).item())


def plot_intensity_hists(orig01: torch.Tensor, recons: dict[str, torch.Tensor], b: int = 0):
    """单张图：原始 vs 各神经元重建的像素强度直方图（展平 RGB）"""
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    o = orig01[b].reshape(-1).numpy()
    bins = np.linspace(0, 1, 60)
    for ax, (name, r) in zip(axes, recons.items()):
        rr = r[b].reshape(-1).detach().cpu().numpy()
        ax.hist(o, bins=bins, alpha=0.45, label="original", color="#333", density=True)
        ax.hist(rr, bins=bins, alpha=0.45, label=f"{name} recon", color="#d95f02", density=True)
        ax.set_title(f"{name}  ($T={T}$)")
        ax.set_xlabel("intensity")
        ax.set_ylabel("density")
        ax.legend(fontsize=8)
    plt.suptitle("原始 vs 重建 强度分布（单样本）", y=1.02)
    plt.tight_layout()
    plt.show()


def plot_recon_montage(orig01: torch.Tensor, recons: dict[str, torch.Tensor], b: int = 0):
    """横向：原图 | LIF | PLIF | MSF"""
    order = ["LIF", "PLIF", "MSF"]
    fig, axes = plt.subplots(1, 1 + len(order), figsize=(3 * (1 + len(order)), 3.2))
    axes[0].imshow(orig01[b].permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Original [0,1]")
    axes[0].axis("off")
    for i, k in enumerate(order, start=1):
        axes[i].imshow(recons[k][b].permute(1, 2, 0).detach().cpu().numpy())
        axes[i].set_title(f"{k} recon")
        axes[i].axis("off")
    plt.suptitle("重建可视化对比（单样本）", y=1.05)
    plt.tight_layout()
    plt.show()


def plot_relative_mse_bar(mse: dict[str, float]):
    """MSF 相对 LIF / PLIF 的 MSE 下降百分比"""
    base_lif = mse["LIF"] + 1e-12
    base_plif = mse["PLIF"] + 1e-12
    red_lif = (base_lif - mse["MSF"]) / base_lif * 100.0
    red_plif = (base_plif - mse["MSF"]) / base_plif * 100.0
    fig, ax = plt.subplots(figsize=(5, 3.5))
    cats = ["vs LIF", "vs PLIF"]
    vals = [red_lif, red_plif]
    cols = ["#1b9e77", "#7570b3"]
    ax.bar(cats, vals, color=cols)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel("MSE 相对减少 (%)\n(MSF 相对基线)")
    ax.set_title("重建误差：MSF 相对基线神经元的改善（batch 平均 MSE）")
    for i, v in enumerate(vals):
        ax.text(i, v + (1 if v >= 0 else -1), f"{v:.2f}%", ha="center", fontsize=10)
    plt.tight_layout()
    plt.show()
    return {"vs_LIF_pct": red_lif, "vs_PLIF_pct": red_plif}


def pseudo_video_strip(orig01: torch.Tensor, recons_msf: torch.Tensor, n_frames: int = 6):
    """将 batch 前 n 帧当作时间轴，拼成条带（示意视频）"""
    n = min(n_frames, orig01.size(0))
    fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 4.5))
    for t in range(n):
        axes[0, t].imshow(orig01[t].permute(1, 2, 0).cpu().numpy())
        axes[0, t].set_title(f"frame {t}")
        axes[0, t].axis("off")
        axes[1, t].imshow(recons_msf[t].permute(1, 2, 0).detach().cpu().numpy())
        axes[1, t].set_title("MSF recon")
        axes[1, t].axis("off")
    plt.suptitle("示意：batch 内连续样本作为“视频帧” | 上行原图 下行 MSF 重建", y=1.02)
    plt.tight_layout()
    plt.show()


def run_pipeline_on_batch(images: torch.Tensor):
    """images: [B,3,H,W] 已 Normalize"""
    orig01 = denormalize_to_01(images)
    x_seq = first_spike_coding(orig01, T=T).to(DEVICE)
    layers = build_neuron_layers(DEVICE)
    recons = {}
    mse = {}
    for name, layer in layers.items():
        out = forward_neuron(layer, x_seq)
        recons[name] = spike_rate_recon(out).cpu()
        mse[name] = mse_vs_original(orig01.cpu(), recons[name])
    return orig01.cpu(), recons, mse


print("REPO:", REPO, "DEVICE:", DEVICE, "T:", T)

## 1. 加载数据（默认 Kaggle；本地改 `USE_KAGGLE = False`）

In [ ]:
import os

# ----- Kaggle：与 kaggle_plantvillage_snn.ipynb 一致，填你的 Input 名称 -----
USE_KAGGLE = True  # 本地 Hugging Face 改为 False

if USE_KAGGLE:
    from data.kaggle_dataloader import get_kaggle_dataloaders

    # 方式 A：仅子路径（位于 /kaggle/input 下）
    INPUT_NAME = "datasets/abdallahalidev/plantvillage-dataset"  # 改成你 Add Data 后的文件夹名
    DATA_ROOT = os.path.join("/kaggle/input", INPUT_NAME)
    # 方式 B：已是绝对路径时可直接赋值覆盖：
    # DATA_ROOT = "/kaggle/input/xxx/plantvillage/color"

    if "AUTO_FIND_SUBDIR" not in globals():
        AUTO_FIND_SUBDIR = True

    train_loader, _, num_classes = get_kaggle_dataloaders(
        DATA_ROOT,
        image_size=224,
        batch_size=16,
        num_workers=2,
        val_ratio=0.2,
        seed=42,
        auto_find_subdir=AUTO_FIND_SUBDIR,
    )
else:
    from data.dataloader import get_dataloaders

    train_loader, _, num_classes = get_dataloaders(batch_size=16, num_workers=0, image_size=224)

if USE_KAGGLE:
    print("DATA_ROOT:", DATA_ROOT)
else:
    print("DATA_ROOT: HF")
print("num_classes:", num_classes, "batches:", len(train_loader))

## 2. 预览：每行 `NUM_COLS` 张，共 `NUM_ROWS` 行

In [ ]:
NUM_COLS = 4
NUM_ROWS = 2

images, labels = next(iter(train_loader))
preview_dataset_grid(images, labels, num_cols=NUM_COLS, num_rows=NUM_ROWS, title_prefix="sample")

## 3. 图 1：原始 vs 重建 强度分布（LIF / PLIF / MSF，$T=4$）

使用上一 cell 同一 batch；直方图为 **batch 中第 0 张图**。

In [ ]:
orig01, recons, mse = run_pipeline_on_batch(images)
print("Batch 平均 MSE (相对 [0,1] 原图):", {k: round(v, 6) for k, v in mse.items()})
plot_intensity_hists(orig01, recons, b=0)

## 4. 图 2：MSF 相对 LIF/PLIF 的 MSE 下降百分比 + 重建条带示意

- **百分比**：\((\mathrm{MSE}_{\mathrm{base}} - \mathrm{MSE}_{\mathrm{MSF}}) / \mathrm{MSE}_{\mathrm{base}} \times 100\%\)。若 MSF 更高则为负，属正常现象（单层脉冲解码无监督、非优化目标）。
- **条带**：batch 内前 6 张图当作“帧”，上行原图、下行 **MSF** 重建。

In [ ]:
stats = plot_relative_mse_bar(mse)
print(stats)
pseudo_video_strip(orig01, recons["MSF"], n_frames=6)

## 5. 单张图：原图与三种重建并排

In [ ]:
plot_recon_montage(orig01, recons, b=0)